In [1]:
import os 
from skimage.io import imread, imsave
from PIL import Image
import numpy as np
from skimage.filters import gaussian
import cv2 as cv

## Define functions to apply CLAHE to your dataset

In [2]:
def CLAHE_single_image(imgpath, tilesize = (12,12), cliplimit=25.0):
    """
    This functions takes the path `imgpath` to an image as input, and
    output a CLAHE-corrected version of the image in the same folder,
    in the format `channel_clahe.tiff`
    """
    
    # Load original MIBI image
    img_float = imread(imgpath)
    img = np.round(img_float*255).astype("uint8")

    # Stretch histogram
    equ = cv.equalizeHist(img)

    # create a CLAHE object (Arguments are optional).
    clahe = cv.createCLAHE(clipLimit=cliplimit, tileGridSize=tilesize)
    cl = clahe.apply(equ)

    tstr = imgpath.split(".")
    outputpath = '.'.join(tstr[:-1]) + '_clahe.' + tstr[-1]
    imsave(outputpath, cl)
    
    return cl

In [4]:
def CLAHE_RGB(chanR, chanG, chanB, fovs = ["."], tilesize = (12,12), cliplimit=25.0):
    """
    This functions creates RGB images for FOVs in folders `fovs`,
    combining the channels provided in `chanR`, `chanG, and `chanB`
    after CLAHE correction of individual input images.
    """  
    for fov in fovs:
        # Make sure output folder exists
        if not os.path.exists(f"/notebooks/data/sampling/{fov}"):
            os.makedirs(f"/notebooks/data/sampling/{fov}")
        outputpath = f"/notebooks/data/sampling/{fov}/RGB.tiff"
        
        claheR = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanR]
        claheR = np.max(claheR, axis = 0)
        claheG = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanG]
        claheG = np.max(claheG, axis = 0)
        claheB = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanB]
        claheB = np.max(claheB, axis = 0)
        
        rgb = np.stack([claheR, claheG, claheB], axis = -1)
        imsave(outputpath, rgb)

In [5]:
# To generate CLAHE corrected images (also calculated in next step)

# Patients with >=2 tumor core and >=2 invasive front FOVs
validation_patients = [
'CRC-IFN_ACCC_10157',
'CRC-IFN_ACCC_1146',
'CRC-IFN_ACCC_13897',
'CRC-IFN_ACCC_15714',
'CRC-IFN_ACCC_16920',
'CRC-IFN_ACCC_17481',
'CRC-IFN_ACCC_17578',
'CRC-IFN_ACCC_245',
'CRC-IFN_ACCC_2775',
'CRC-IFN_ACCC_3897',
'CRC-IFN_ACCC_4267',
'CRC-IFN_ACCC_5006',
'CRC-IFN_ACCC_5267',
'CRC-IFN_ACCC_5413',
'CRC-IFN_ACCC_589',
'CRC-IFN_ACCC_6341',
'CRC-IFN_ACCC_6802',
'CRC-IFN_ACCC_7140',
'CRC-IFN_ACCC_7148',
'CRC-IFN_ACCC_7451',
'CRC-IFN_ACCC_7548',
'CRC-IFN_ACCC_7701',
'CRC-IFN_ACCC_8256',
'CRC-IFN_ACCC_8305',
'CRC-IFN_ACCC_8373',
'CRC-IFN_ACCC_9600'
]

# CLAHE_RGB(
#     chanR = ["HH3_dsDNA.tiff"],
#     chanG = ["HLA1.tiff"],
#     chanB = ["HLA1.tiff", "GLUT1.tiff", "ASCT2CAL33_ASCT2D7C12.tiff"],
#     fovs = [f"/data/{x}" 
#             for x in os.listdir("/data/") if x[0] != '.' and any(y in x for y in validation_patients)],)

## Define functions for segmentation and cell intensity extraction
Warning: the segmentation masks are saved as UINT16, so there will be issues if more than 65K objects are segmented on an image (unrealistic on MIBI images).

In [6]:
import pandas as pd
import time
from copy import copy

import logging
clog = logging.getLogger("cellpose")
clog.propagate = False
from cellpose import models

from skimage.measure import regionprops_table
from tqdm import tqdm

In [13]:
# Not quantitative channels to extract
excluded_chans = [
    'chan_39.tiff',
    'RGB.tiff',
    'cellpose_filtered.tiff',
    "chan_39.tiff",
    "chan_48.tiff",
    "chan_69.tiff",
    "chan_71.tiff",
    "Fe.tiff",
    "Noodle.tiff",
    "segmentation.tiff",
]

def CLAHE_cellpose_extract(chanR, chanG, chanB, 
                           fovs = ["."], 
                           tilesize = (12,12), 
                           cliplimit=25.0,
                           seg_channels = [3,1],
                           seg_model_type = "TN2",
                           seg_diameter = 25,
                           seg_filter_nuclear_pixel = 80,
                           chans_to_extract = True, # Default to all in folder
                          ):
    """
    This functions creates RGB images for FOVs in folders `fovs`,
    combining the channels provided in `chanR`, `chanG, and `chanB`
    after CLAHE correction of individual input images. These images
    are used for segmentation using CellPose, and the intensities for
    all channels are extracted in a cell intensity table.    
    """  
    cpmodel = models.CellposeModel(gpu=False, model_type=seg_model_type)                            
    for fov in tqdm(fovs):
        print(f"{time.ctime()} - Processing {fov}.", flush=True)
        if not os.path.exists(f"/notebooks/data/sampling/{fov}"):
            os.makedirs(f"/notebooks/data/sampling/{fov}")  
        outputpath = f"/notebooks/data/sampling/{fov}"
        
        claheR = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanR]
        claheR = np.max(claheR, axis = 0)
        claheG = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanG]
        claheG = np.max(claheG, axis = 0)
        claheB = [CLAHE_single_image(f"{fov}/{chan}", tilesize = tilesize, cliplimit=cliplimit) for chan in chanB]
        claheB = np.max(claheB, axis = 0)
        
        rgb = np.stack([claheR, claheG, claheB], axis = -1)
        imsave(outputpath+"/RGB.tiff", rgb)
        
        segmentation = cpmodel.eval(
            rgb, diameter=seg_diameter, channels=seg_channels, resample=False
        )

        # Pixels per object
        pixels_per_nucleus = np.unique(segmentation[0], return_counts=True)
        
        nucleus_ids = pixels_per_nucleus[0][1:][pixels_per_nucleus[1][1:] > seg_filter_nuclear_pixel] # Discard speckles
        filterimg = copy(segmentation[0])
        filterimg[np.isin(segmentation[0], nucleus_ids) == False] = 0
        imsave(f"{outputpath}/cellpose_filtered.tiff", 
                         filterimg)

        print(f"{time.ctime()} - {len(nucleus_ids)} segmentation masks kept. Extracting features.", flush=True)
        
        # Morphological features
        props = regionprops_table(filterimg, properties=('centroid',
                                                 'eccentricity',
                                                 'perimeter',
                                                 'convex_area',
                                                 'area',
                                                 'axis_major_length',
                                                 'axis_minor_length',
                                                 'label'))

        if chans_to_extract == True:
            # Set to all chans in (first) dir
            chans_to_extract = [x for x in os.listdir(fov) if x[0] != '.' 
                                        and x not in excluded_chans]
                    
        # We want a DF with rows = nuclei and columns = intensities
        markers = pd.DataFrame({c: [-1 for _ in props["label"]] for c in chans_to_extract})
        markers.set_index(props["label"], inplace = True)

        for chan in chans_to_extract:
            # Read a channel
            chan_data = imread(f"{fov}/{chan}")
            # Look at each cell in table
            for n in props["label"]:
                n_mask = filterimg == n
                markers.loc[n, chan] = np.mean(chan_data[n_mask])
                
        # Add morphology
        props = pd.DataFrame(props)
        props.set_index(props["label"], inplace = True)
        markers = pd.concat([markers, props], axis=1)
                
        # Export marker table
        markers.to_csv(f"{outputpath}/cell_table.csv")

In [ ]:
CLAHE_cellpose_extract(
    chanR = ["HH3_dsDNA.tiff"],
    chanG = ["HLA1.tiff"],
    chanB = ["HLA1.tiff", "GLUT1.tiff", "ASCT2CAL33_ASCT2D7C12.tiff"],
    fovs = [f"/data/{x}" 
            for x in os.listdir("/data/") if x[0] != '.' and any(y in x for y in validation_patients)],)

  0%|                                                | 0/156 [00:00<?, ?it/s]

Fri Oct  9 16:55:16 2026 - Processing /data/CRC-IFN_ACCC_7701_IM_1.


Fri Oct  9 16:55:53 2026 - 3546 segmentation masks kept. Extracting features.


  1%|▏                                    | 1/156 [02:03<5:18:52, 123.44s/it]

Fri Oct  9 16:57:19 2026 - Processing /data/CRC-IFN_ACCC_15714_IM_3.
Fri Oct  9 16:57:53 2026 - 5038 segmentation masks kept. Extracting features.


...

Fri Oct  9 17:08:30 2026 - Processing /data/CRC-IFN_ACCC_3897_TC_1.
Fri Oct  9 17:09:05 2026 - 4127 segmentation masks kept. Extracting features.


  4%|█▋                                   | 7/156 [15:25<5:25:57, 131.26s/it]

Fri Oct  9 17:10:42 2026 - Processing /data/CRC-IFN_ACCC_8256_TC_3.
Fri Oct  9 17:11:15 2026 - 4245 segmentation masks kept. Extracting features.


  5%|█▉                                   | 8/156 [17:39<5:25:44, 132.06s/it]

Fri Oct  9 17:12:56 2026 - Processing /data/CRC-IFN_ACCC_7701_TC_1.
Fri Oct  9 17:13:39 2026 - 4602 segmentation masks kept. Extracting features.
